# 3. Partición común y base de modelado

La prueba de DeLong y las comparaciones pareadas requieren que todos los modelos se evalúen sobre exactamente las mismas observaciones. En este capítulo se crea una sola partición 80/20, estratificada por clase y con semilla fija. La asignación se guarda en Parquet y será leída tanto por scikit-learn como por PySpark.

In [1]:
from pathlib import Path
import json
import platform

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split

candidates = [
    Path('../accepted_2007_to_2018Q4.csv'),
    Path('../../accepted_2007_to_2018Q4.csv'),
    Path('Tarea1/accepted_2007_to_2018Q4.csv'),
]
CSV_PATH = next((p.resolve() for p in candidates if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError('No se encontró accepted_2007_to_2018Q4.csv')

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
RESULTS_DIR = PROJECT_DIR / 'results' / 'preprocessing'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
print(f'Python: {platform.python_version()} | pandas: {pd.__version__} | sklearn: {sklearn.__version__}')

Python: 3.12.14 | pandas: 3.0.6 | sklearn: 1.9.1


## 3.1 Variables y control de fuga de información

La base conserva variables disponibles al momento de la solicitud u originación. Se excluyen explícitamente resultados posteriores como pagos acumulados, recuperaciones, saldo pendiente, fecha o monto del último pago y estado de hardship. También se excluyen campos de texto libre o identificadores que no deben actuar como predictores.

In [2]:
numeric_features = [
    'loan_amnt', 'int_rate', 'fico_range_high', 'annual_inc', 'dti',
]
categorical_features = [
    'emp_length', 'purpose', 'home_ownership', 'addr_state',
    'verification_status', 'grade', 'term',
]
metadata_cols = ['id', 'loan_status', 'issue_d']
selected_cols = metadata_cols + numeric_features + categorical_features

post_outcome_examples = [
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries',
    'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
    'next_pymnt_d', 'last_fico_range_high', 'last_fico_range_low',
    'debt_settlement_flag', 'settlement_status', 'settlement_amount',
]

feature_catalog = pd.DataFrame({
    'variable': numeric_features + categorical_features,
    'tipo': ['numérica'] * len(numeric_features) + ['categórica'] * len(categorical_features),
    'momento': 'Disponible al originarse el préstamo',
})
display(feature_catalog)
feature_catalog.to_csv(RESULTS_DIR / 'catalogo_variables_modelado.csv', index=False, encoding='utf-8-sig')

,variable,tipo,momento
0,loan_amnt,numérica,Disponible al originarse el préstamo
1,int_rate,numérica,Disponible al originarse el préstamo
2,fico_range_high,numérica,Disponible al originarse el préstamo
3,annual_inc,numérica,Disponible al originarse el préstamo
4,dti,numérica,Disponible al originarse el préstamo
5,emp_length,categórica,Disponible al originarse el préstamo
6,purpose,categórica,Disponible al originarse el préstamo
7,home_ownership,categórica,Disponible al originarse el préstamo
8,addr_state,categórica,Disponible al originarse el préstamo
9,verification_status,categórica,Disponible al originarse el préstamo


El catálogo reúne cinco variables numéricas y siete categóricas, todas disponibles al originarse el préstamo. El identificador, el estado de pago y la fecha se conservan como metadatos para enlazar y describir registros, pero no se incluyen como predictores. Así se reduce el riesgo de que el modelo use información creada después del resultado que intenta anticipar.

## 3.2 Carga completa de la población elegible

In [3]:
target_map = {'Fully Paid': 0, 'Charged Off': 1}
parts = []

for chunk in pd.read_csv(CSV_PATH, usecols=selected_cols, chunksize=150_000, low_memory=False):
    numeric_id = chunk['id'].astype('string').str.fullmatch(r'\d+', na=False)
    eligible = numeric_id & chunk['loan_status'].isin(target_map)
    if eligible.any():
        part = chunk.loc[eligible].copy()
        part['id'] = part['id'].astype('string')
        part['default'] = part['loan_status'].map(target_map).astype('int8')
        parts.append(part)

model_base = pd.concat(parts, ignore_index=True)
del parts

assert len(model_base) == 1_345_310, 'Cambió el número esperado de préstamos elegibles.'
assert model_base['id'].is_unique, 'El identificador id debe ser único.'
assert set(model_base['default'].unique()) == {0, 1}

print(f'Filas: {len(model_base):,}')
print(f'IDs únicos: {model_base.id.nunique():,}')
display(model_base.head())

Filas: 1,345,310


IDs únicos: 1,345,310


,id,loan_amnt,term,int_rate,grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,fico_range_high,default
0,68407277,3600.0,36 months,13.99,C,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,debt_consolidation,PA,5.91,679.0,0
1,68355089,24700.0,36 months,11.99,C,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,small_business,SD,16.06,719.0,0
2,68341763,20000.0,60 months,10.78,B,10+ years,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,home_improvement,IL,10.78,699.0,0
3,68476807,10400.0,60 months,22.45,F,3 years,MORTGAGE,104433.0,Source Verified,Dec-2015,Fully Paid,major_purchase,PA,25.37,699.0,0
4,68426831,11950.0,36 months,13.44,C,4 years,RENT,34000.0,Source Verified,Dec-2015,Fully Paid,debt_consolidation,GA,10.20,694.0,0


La carga conserva los 1.345.310 préstamos elegibles y confirma que existen 1.345.310 identificadores únicos. La comprobación descarta duplicados por `id` y verifica que estén presentes ambas clases. La tabla de cinco filas es solo una vista de ejemplo; el ajuste posterior usa la base completa.

## 3.3 Creación de la partición estratificada

In [4]:
indices = np.arange(len(model_base))
train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=model_base['default'].to_numpy(),
)

split_values = np.full(len(model_base), 'train', dtype=object)
split_values[test_idx] = 'test'

common_split = model_base[['id', 'default']].copy()
common_split['split'] = split_values

assert len(train_idx) + len(test_idx) == len(model_base)
assert np.intersect1d(train_idx, test_idx).size == 0
assert common_split['id'].is_unique
assert not common_split['split'].isna().any()

split_summary = (
    common_split.groupby(['split', 'default'])
    .size().rename('n').reset_index()
)
split_summary['porcentaje_dentro_split'] = (
    100 * split_summary['n'] / split_summary.groupby('split')['n'].transform('sum')
)
display(split_summary)
split_summary.to_csv(RESULTS_DIR / 'resumen_particion.csv', index=False, encoding='utf-8-sig')

,split,default,n,porcentaje_dentro_split
0,test,0,215350,80.037315
1,test,1,53712,19.962685
2,train,0,861401,80.037408
3,train,1,214847,19.962592


La partición contiene 1.076.248 observaciones de entrenamiento y 269.062 de prueba. En ambos grupos, cerca de 80,04 % son préstamos pagados y 19,96 % castigados; esa cercanía confirma que la estratificación conservó la distribución del target. Ningún registro se comparte entre los dos grupos: el entrenamiento sirve para aprender y seleccionar modelos, y la prueba queda reservada para la evaluación final.

## 3.4 Persistencia y comprobación de reproducibilidad

In [5]:
split_path = PROCESSED_DIR / 'common_split.parquet'
base_path = PROCESSED_DIR / 'lending_club_model_base.parquet'

common_split.to_parquet(split_path, index=False, compression='snappy')
model_base_with_split = model_base.merge(
    common_split[['id', 'split']], on='id', how='left', validate='one_to_one'
)
model_base_with_split.to_parquet(base_path, index=False, compression='snappy')

# Reapertura: valida los artefactos realmente escritos, no solo los objetos en memoria.
split_check = pd.read_parquet(split_path)
base_check = pd.read_parquet(base_path, columns=['id', 'default', 'split'])

assert split_check.equals(common_split)
assert len(base_check) == len(model_base)
assert base_check['id'].is_unique
assert base_check['split'].value_counts().to_dict() == common_split['split'].value_counts().to_dict()

manifest = {
    'source_file': CSV_PATH.name,
    'eligible_statuses': target_map,
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'rows_total': int(len(common_split)),
    'rows_train': int((common_split['split'] == 'train').sum()),
    'rows_test': int((common_split['split'] == 'test').sum()),
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'split_file': split_path.name,
    'model_base_file': base_path.name,
}
with open(PROCESSED_DIR / 'manifest.json', 'w', encoding='utf-8') as stream:
    json.dump(manifest, stream, ensure_ascii=False, indent=2)

print(f'Partición guardada: {split_path} ({split_path.stat().st_size / 1024**2:.2f} MiB)')
print(f'Base guardada: {base_path} ({base_path.stat().st_size / 1024**2:.2f} MiB)')
display(split_check.head())

Partición guardada: C:\Users\Valentina Figueroa\OneDrive - Universidad del Norte\PERSONAL\MAESTRÍA ANALÍTICA DE DATOS\Machine Learning\Tarea1\proyecto\data\processed\common_split.parquet (7.88 MiB)
Base guardada: C:\Users\Valentina Figueroa\OneDrive - Universidad del Norte\PERSONAL\MAESTRÍA ANALÍTICA DE DATOS\Machine Learning\Tarea1\proyecto\data\processed\lending_club_model_base.parquet (20.76 MiB)


,id,default,split
0,68407277,0,train
1,68355089,0,test
2,68341763,0,train
3,68476807,0,test
4,68426831,0,train


Los archivos Parquet guardan la asignación y la base reducida en formatos compactos (aproximadamente 7,88 MiB y 20,76 MiB). Después de escribirlos, se vuelven a abrir y se comprueban fila por fila, junto con el número de identificadores y los tamaños de cada partición. Esto verifica que los entornos posteriores usarán los mismos registros y que el artefacto persistido coincide con el objeto original.

## 3.5 Uso obligatorio en ambos entornos

**scikit-learn:** leer `lending_club_model_base.parquet` y separar mediante la columna `split`. Los imputadores, codificadores y escaladores se ajustarán solo sobre `split == 'train'`.

**PySpark:** leer la misma base Parquet o unir el CSV con `common_split.parquet` mediante `id`, y filtrar por `split`. No se utilizará `randomSplit`, porque produciría observaciones diferentes.

La base Parquet conserva todas las filas elegibles. La reducción de columnas responde a la selección documentada de predictores y a la exclusión de fuga de información; no es muestreo.